In [5]:
import os
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from collections import Counter
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

# ==========================================
# 1. ROBUST DATASETS
# ==========================================
SHARED_LABEL_MAP = {
    'E': 0, 'W': 1, 'R': 2, 'J': 3, 
    'L': 4, 'S': 5, 'C': 6, 'H': 7
}

class SHARPTrainDataset(Dataset):
    def __init__(self, root_dir, target_sets=['S1'], window_size=340, step_size=170):
        self.samples =[]
        self.label_map = SHARED_LABEL_MAP
        
        print(f"Loading training data for sets: {target_sets}...")
        for root, _, files in os.walk(root_dir):
            folder_name = os.path.basename(root)
            if not any(folder_name.startswith(ts) for ts in target_sets):
                continue
                
            for file in files:
                if file.endswith(".txt"):
                    try:
                        raw_code = file.split('_')[1].upper()
                        # Strip digits (e.g., C1 -> C)
                        code = ''.join([char for char in raw_code if not char.isdigit()])
                        
                        if code in self.label_map:
                            label = self.label_map[code]
                            # NO NORMALIZATION! Load raw float32
                            data = np.load(os.path.join(root, file), allow_pickle=True).astype(np.float32)
                            
                            for start in range(0, data.shape[0] - window_size + 1, step_size):
                                window = data[start : start + window_size, :]
                                self.samples.append((window, label))
                    except Exception as e:
                        continue
        print(f"Loaded {len(self.samples)} train samples.")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        window, label = self.samples[idx]
        return torch.from_numpy(window).unsqueeze(0), torch.tensor(label, dtype=torch.long)


class SHARPTestDataset(Dataset):
    def __init__(self, root_dir, target_sets, window_size=340, step_size=170):
        self.samples =[]
        self.label_map = SHARED_LABEL_MAP

        for root, _, files in os.walk(root_dir):
            folder_name = os.path.basename(root)
            if not any(folder_name.startswith(ts) for ts in target_sets):
                continue

            for file in files:
                if file.endswith("stream_0.txt"): 
                    try:
                        raw_code = file.split('_')[1].upper()
                        code = ''.join([char for char in raw_code if not char.isdigit()])
                        
                        if code in self.label_map:
                            label = self.label_map[code]
                            base_path = os.path.join(root, file.replace("_stream_0.txt", ""))
                            
                            streams =[]
                            for i in range(4):
                                stream_file = f"{base_path}_stream_{i}.txt"
                                if not os.path.exists(stream_file): 
                                    stream_file = f"{base_path}_stream_0.txt"
                                
                                # NO NORMALIZATION!
                                streams.append(np.load(stream_file, allow_pickle=True).astype(np.float32))
                            
                            length = streams[0].shape[0]
                            for start in range(0, length - window_size + 1, step_size):
                                end = start + window_size
                                self.samples.append(([s[start:end, :] for s in streams], label))
                    except Exception as e:
                        print(f"Skipped {file} due to error: {e}")
                        continue

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        streams, label = self.samples[idx]
        tensors =[torch.from_numpy(s).unsqueeze(0) for s in streams]
        return tensors[0], tensors[1], tensors[2], tensors[3], torch.tensor(label, dtype=torch.long)

# ==========================================
# 2. ARCHITECTURE & FUSION
# ==========================================
class SHARP_Original_Architecture(nn.Module):
    def __init__(self, num_classes=8, dropout=0.2): 
        super().__init__()
        
        self.input_norm = nn.BatchNorm2d(1) 
        self.branch1 = nn.MaxPool2d(kernel_size=2, stride=2)
        # (Optional but recommended) You can also add BatchNorm after your Conv2d layers 
        # to further stabilize training. Standard practice is Conv2d -> BatchNorm2d -> ReLU
        self.branch2 = nn.Sequential(
            nn.Conv2d(1, 5, 2, 2), 
            nn.BatchNorm2d(5), 
            nn.ReLU()
        )
        self.branch3 = nn.Sequential(
            nn.Conv2d(1, 3, 1, 1), 
            nn.BatchNorm2d(3),
            nn.ReLU(),
            nn.ZeroPad2d((0, 1, 0, 1)), 
            nn.Conv2d(3, 6, 2, 1), 
            nn.BatchNorm2d(6),
            nn.ReLU(),
            nn.Conv2d(6, 9, 4, 2, 1), 
            nn.BatchNorm2d(9),
            nn.ReLU()
        )
        self.concat_conv = nn.Sequential(
            nn.Conv2d(15, 3, 1, 1), 
            nn.BatchNorm2d(3),
            nn.ReLU()
        )
        self.flatten = nn.Flatten()
        self.dropout = nn.Dropout(dropout)
        self.dense = nn.Linear(25500, num_classes)

    def forward(self, x):
        # 2. Pass the raw input through the normalizer FIRST
        x = self.input_norm(x)
        
        x = torch.cat([self.branch1(x), self.branch2(x), self.branch3(x)], dim=1)
        x = self.concat_conv(x)
        x = self.dense(self.dropout(self.flatten(x)))
        return x

def soft_decision_fusion(logits_list):
    probs_list =[torch.softmax(logits, dim=1) for logits in logits_list]
    avg_probs = torch.mean(torch.stack(probs_list), dim=0)
    return torch.argmax(avg_probs, dim=1)

# ==========================================
# 3. TRAINING & EVALUATION LOOP
# ==========================================
if __name__ == "__main__":
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Running on: {device}")

    # --- Load Training Data ---
    train_dataset = SHARPTrainDataset("doppler_traces", target_sets=['S1']) 
    train_size = int(0.75 * len(train_dataset)) 
    val_size = len(train_dataset) - train_size
    train_set, val_set = torch.utils.data.random_split(train_dataset, [train_size, val_size])

    train_loader = DataLoader(train_set, batch_size=64, shuffle=True)
    val_loader = DataLoader(val_set, batch_size=128, shuffle=False)
    
    # --- Init Model ---
    model = SHARP_Original_Architecture(num_classes=8, dropout=0.2).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.AdamW(model.parameters(), lr=0.001, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=30)

    best_val_acc = 0.0
    epochs = 30
    
    for epoch in range(epochs):
        model.train()
        train_correct, train_total = 0, 0
        
        for inputs, targets in train_loader:
            inputs, targets = inputs.to(device), targets.to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            loss.backward()
            optimizer.step()
            
            _, predicted = torch.max(outputs, 1)
            train_total += targets.size(0)
            train_correct += (predicted == targets).sum().item()

        scheduler.step()

        model.eval()
        val_correct, val_total = 0, 0
        with torch.no_grad():
            for inputs, targets in val_loader:
                inputs, targets = inputs.to(device), targets.to(device)
                outputs = model(inputs)
                _, predicted = torch.max(outputs, 1)
                val_total += targets.size(0)
                val_correct += (predicted == targets).sum().item()

        val_acc = val_correct / val_total
        print(f"Epoch {epoch+1}/{epochs} | Train Acc: {train_correct/train_total:.4f} | Val Acc: {val_acc:.4f}")

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save(model.state_dict(), "sharp_best_weights.pth")

    # --- Test on S2 through S7 ---
    print("\n--- Starting SHARP Decision Fusion Testing ---")
    model.load_state_dict(torch.load("sharp_best_weights.pth"))
    model.eval()
    
    test_sets =['S2', 'S3', 'S4', 'S5', 'S6','S7']
    results = {}

    with torch.no_grad():
        for test_set_id in test_sets:
            print(f"Evaluating {test_set_id}...")
            test_dataset = SHARPTestDataset("doppler_traces", target_sets=[test_set_id])
            
            if len(test_dataset) == 0:
                print(f"  -> No data found for {test_set_id}.")
                continue
                
            test_loader = DataLoader(test_dataset, batch_size=128, shuffle=False)
            test_correct, test_total = 0, 0
            
            for x1, x2, x3, x4, targets in test_loader:
                x1, x2, x3, x4 = x1.to(device), x2.to(device), x3.to(device), x4.to(device)
                targets = targets.to(device)
                
                logits =[model(x1), model(x2), model(x3), model(x4)]
                final_preds = soft_decision_fusion(logits)
                
                test_total += targets.size(0)
                test_correct += (final_preds == targets).sum().item()
            
            if test_total > 0:
                accuracy = test_correct / test_total
                results[test_set_id] = accuracy
                print(f"  -> Accuracy for {test_set_id}: {accuracy * 100:.2f}%\n")
            else:
                print(f"  -> Skipping accuracy calculation, 0 valid test batches.")

    print("\n" + "="*40)
    print(" SHARP GENERALIZATION REPORT (TABLE 3)")
    print("="*40)
    for folder, acc in results.items():
        print(f" Test Set {folder} Accuracy : {acc * 100:.2f}%")
    print("="*40)

Running on: cuda
Loading training data for sets: ['S1']...
Loaded 10436 train samples.
Epoch 1/30 | Train Acc: 0.2873 | Val Acc: 0.3622
Epoch 2/30 | Train Acc: 0.4299 | Val Acc: 0.3695
Epoch 3/30 | Train Acc: 0.4704 | Val Acc: 0.3637
Epoch 4/30 | Train Acc: 0.5243 | Val Acc: 0.3373
Epoch 5/30 | Train Acc: 0.5316 | Val Acc: 0.3162
Epoch 6/30 | Train Acc: 0.5826 | Val Acc: 0.4891
Epoch 7/30 | Train Acc: 0.6207 | Val Acc: 0.4998
Epoch 8/30 | Train Acc: 0.6235 | Val Acc: 0.4553
Epoch 9/30 | Train Acc: 0.6625 | Val Acc: 0.3492
Epoch 10/30 | Train Acc: 0.6966 | Val Acc: 0.5079
Epoch 11/30 | Train Acc: 0.6992 | Val Acc: 0.4787
Epoch 12/30 | Train Acc: 0.7076 | Val Acc: 0.4941
Epoch 13/30 | Train Acc: 0.7148 | Val Acc: 0.4772
Epoch 14/30 | Train Acc: 0.7479 | Val Acc: 0.4895
Epoch 15/30 | Train Acc: 0.7432 | Val Acc: 0.4764
Epoch 16/30 | Train Acc: 0.7553 | Val Acc: 0.4094
Epoch 17/30 | Train Acc: 0.7676 | Val Acc: 0.4780
Epoch 18/30 | Train Acc: 0.7779 | Val Acc: 0.4895
Epoch 19/30 | Train Ac